# 08 — Core proof-of-concept visualisations

This reporting notebook reads the saved outputs from Notebook 07 and creates a compact set of figures. It does not recompute the model.

These figures are useful for quick checks and presentations; the later Notebook 13 provides the full narrative synthesis.

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import osmnx as ox
import pandas as pd

ACCESS_DIR = OUTPUTS / "accessibility"

sensitivity = pd.read_csv(
    ACCESS_DIR / "accessibility_sensitivity.csv"
)

access = pd.read_csv(
    ACCESS_DIR / "origin_accessibility_lambda_005.csv"
)

## 1. Sensitivity of accessibility to λ

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(
    sensitivity["lambda"],
    sensitivity["mean_access_loss_pct"],
    marker="o",
)

ax.set_xlabel("Thermal penalty parameter λ")
ax.set_ylabel("Mean accessibility loss (%)")
ax.set_title("Accessibility sensitivity to thermal impedance")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Normal vs heat-conditioned accessibility

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(
    access["normal_access"],
    access["thermal_access"],
    alpha=0.75,
)

limit = max(
    access["normal_access"].max(),
    access["thermal_access"].max(),
)

ax.plot([0, limit], [0, limit], linestyle="--")

ax.set_xlabel("Normal reachable transit stops")
ax.set_ylabel("Thermal reachable transit stops")
ax.set_title("Normal vs heat-conditioned accessibility")

plt.tight_layout()
plt.show()

## 3. Spatial accessibility loss

In [ ]:
graph = ox.load_graphml(
    DATA_INTERIM / "sevilla_walk_thermal.graphml"
)

nodes, edges = ox.graph_to_gdfs(graph)
nodes = nodes.reset_index()
nodes["osmid"] = nodes["osmid"].astype(str)

access["origin"] = access["origin"].astype(str)

origins = nodes[
    nodes["osmid"].isin(access["origin"])
].merge(
    access,
    left_on="osmid",
    right_on="origin",
    how="inner",
)

fig, ax = plt.subplots(figsize=(8, 8))

edges.plot(ax=ax, linewidth=0.4, alpha=0.25)

origins.plot(
    ax=ax,
    column="access_loss_pct",
    markersize=45,
    legend=True,
)

ax.set_title("Heat-conditioned accessibility loss by sampled origin")
ax.set_axis_off()

plt.tight_layout()
plt.show()